In [10]:
!pip install -q pinecone

In [11]:
import os
from getpass import getpass

# Enter your NEW Pinecone API key (after regenerating it)
os.environ["PINECONE_API_KEY"] = getpass("Enter your Pinecone API Key: ")

Enter your Pinecone API Key: ··········


In [12]:
from pinecone import Pinecone

# Initialize Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Connect to the existing index
index = pc.Index("amazon-policies")

print("✅ Successfully connected to the Pinecone index!")

✅ Successfully connected to the Pinecone index!


In [13]:
# Describe index statistics
stats = index.describe_index_stats()
print("📊 Index Statistics:")
print(stats)

📊 Index Statistics:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '185',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 09 Apr 2026 20:54:45 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '6',
                                    'x-pinecone-request-latency-ms': '5',
                                    'x-pinecone-response-duration-ms': '7'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 382}},
 'storageFullness': 0.0,
 'total_vector_count': 382,
 'vector_type': 'dense'}


In [14]:
from sentence_transformers import SentenceTransformer

# Load the same embedding model used during indexing
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def query_pinecone(query, top_k=5):
    # Generate query embedding
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Query the index
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    return results

# Example query
results = query_pinecone("What is the return policy?")

# Display results
for i, match in enumerate(results["matches"], 1):
    print(f"\nResult {i}")
    print(f"Score  : {match['score']:.4f}")
    print(f"Source : {match['metadata'].get('source')}")
    print(f"Page   : {match['metadata'].get('page')}")
    print(f"Text   : {match['metadata'].get('text')[:300]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Result 1
Score  : 0.6470
Source : Amazon.in Returns Policy - Amazon Customer Service.pdf
Page   : 1
Text   : . General Returns Policy In the event customers are found to misuse the return policy by excessively returning, or cancelling or not accepting the orders placed, Amazon reserves the right to warn and/or suspend and/or block and/or terminate such customer accounts, as necessary. . Applicable products

Result 2
Score  : 0.6215
Source : Amazon.in Returns Policy - Amazon Customer Service.pdf
Page   : 1
Text   : Amazon.in Returns Policy Information on return eligibility, timelines and other terms & conditions for items purchased on Amazon.in. Disclaimer: In the event of any discrepancy or conflict, the English version will prevail over the translation. Most items purchased from sellers listed on Amazon.in a

Result 3
Score  : 0.6208
Source : Amazon.in Returns Policy - Amazon Customer Service.pdf
Page   : 6
Text   : Subscriptions In-skill subscription purchases are eligible for a ful